In [ ]:
%%bash

pip install --upgrade 'scikit-learn<1.6.0'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Frame the Problem (15 pts)

Answer the following questions.

1. Define the objective: what are you trying to predict?
2. How should you frame this problem as a machine learning problem?
3. How should performance be measured?

**1.** The objective of this model is to predict the sale price of houses in Ames, Iowa based on their 79 features provided. We need to predict the value of the SalesPrice variable and output the specific Id of the corresponding house.

**2.** Our model will need to learn from the data in the training set (E) to perform the task of predicting the sale prices of the houses (T) given the 79 features. The performance of our model (P) is measured by how low the Root-Mean-Squared-Error (RMSE) of the logarithm of our predicition is compared to the actual sale price.

**3.** As stated, the perforamnce of our model will be based on the RMSE of the logarithm of our prediction. The lower the RMSE is, the more accurate and better the performance is.

# 2. Get the Data (15 pts)

Minimum expectations:

1. Load the data.
2. Check the size and type of data.

In [ ]:
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv",index_col="Id")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv",index_col="Id")
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

* From the above information we can deduce that we have a total of 1460 houses (sample size). Additionally, we have 79 features to base our model on.
* In terms of data-types we have:
  * 3 float64
  * 34 int64
  * 43 objects.

In [ ]:
cat /kaggle/input/house-prices-advanced-regression-techniques/data_description.txt

# 3. Explore the Data (15 pts)

General expectations

1. Discuss the types of features (categorical, ordinal, int/float, bounded/unbounded, text, etc.)
2. What features have missing values?
3. Are there any outliers? If so, how can you tell?
4. What features appear usefulness for the task? Are there any that appear useless? Why?
5. Identify and characterize the target distribution.
6. Visualize the data.
7. Study the correlations between attributes.

In [ ]:
num_fts = train.select_dtypes(include=["int64", "float64"]).columns
print("int/float features: ", num_fts)

obj_fts = train.select_dtypes(include=["object"]).columns
print("catergorical features: ", obj_fts)

**1.** We have 37 features that are numerical (int/floats) and 43 that are objects listed above. In order to know which are ordinal or nominal we look at the data_description.txt file above.
- Examples of ordinal features: OverallQual, OverallCond, ExterQual, BsmtQual, FireplaceQu
- Examples of nominal features: Neighborhood, MSZoning, Exterior1st, HouseStyle, SaleCondition
- We also have features that are bounded within a fixed set of options, eg: OverallQual which ranges from 1-10
- Unbounded features include LotArea which do not have a fixed range

In [ ]:
null_count = train.isnull().sum()
null_count = null_count[null_count > 0]
print(f"Total columns with missing values: {len(null_count)}\n")
print(null_count.sort_values(ascending=False))

**2.** Above shows the number of features with essentially missing entries. 
* For Example:
    * PoolQC has 1453 missing values

In [ ]:
num_fts = [col for col in num_fts if col not in ['Id', 'SalePrice']]

for col in num_fts:
    plt.figure(figsize=(3, 2))
    plt.scatter(train[col], train['SalePrice'])
    plt.title(f'{col}')
    plt.show()

**3.** We can look at the plots above to look for features with extreme outliers. Essentially, we can tell by how far some data points are away from other data points, some examples are:
- LotFrontage
- LotArea
- BsmtFinSF1
- BsmtFinSF2
- 1stFlrSF
- GrvLivArea
- MiscVal

In [ ]:
for col in obj_fts:
    print("--------------------")
    print(train[col].value_counts(dropna=True))

**4.** 

Firstly, I believe we should disregard features with a high missing value count:

* PoolQC (1453)
* MiscFeature (1406)
* Alley (1369)
* Fence (1179)

Secondly, features with not that many variety (little variance) such as :

* Utilites (all but 1 entry is the same)
* Street
* LandSlope
* Condition2
* RoofMatl
* BsmtCond
* Heating

Some features have a linear relationship with price:

* OverallQual
* GrLivArea
* GarageCars
* TotalBsmtSF
* 1stFlrSF
* FullBath
* YearBuilt

In [ ]:
from scipy.stats import skew, kurtosis
sale_col = train["SalePrice"].dropna()
print("Skew:", skew(sale_col))
print("Kurtosis:", kurtosis(sale_col))

plt.figure(figsize=(6,4))
plt.hist(sale_col, bins=50)
plt.show()

In [ ]:
sale_col_log = np.log1p(sale_col)
print("Skew:", skew(sale_col_log))
print("Kurtosis:", kurtosis(sale_col_log)) 

plt.figure(figsize=(6,4))
plt.hist(sale_col_log, bins=40, alpha=0.9)
plt.show()

**5.** From the first plot we can see that the actual sale prices are skweded to the right with a high kurtosis. 
Taking the log of the sale prices gives us a more even distribution which makes it better for training.

In [ ]:
train["SalePrice"] = np.log1p(train["SalePrice"])

In [ ]:
from sklearn.preprocessing import LabelEncoder

#Correlation heatmap of all features used ChatGPT to write the next two code blocks
df_encoded = train.copy()

obj_cols = df_encoded.select_dtypes(include='object').columns
le = LabelEncoder()

for col in obj_cols:
    df_encoded[col] = df_encoded[col].astype(str)
    df_encoded[col] = le.fit_transform(df_encoded[col])

corr_all = df_encoded.corr(method='spearman')

plt.figure(figsize=(16, 12))
plt.imshow(corr_all, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(shrink=.5)
plt.title("Correlation heatmap of all features")
plt.xticks(range(len(corr_all.columns)), corr_all.columns, rotation=90, fontsize=6)
plt.yticks(range(len(corr_all.index)), corr_all.index, fontsize=6)
plt.show()

In [ ]:
# Heatmap of top 30 most correlated features
k = 30
target = 'SalePrice'

corr_target = corr_all[target].drop(target).abs().sort_values(ascending=False)
top_features = corr_target.head(k).index.tolist() + [target]
corr_top = df_encoded[top_features].corr(method='spearman')

plt.figure(figsize=(12, 10))
plt.imshow(corr_top, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(shrink=.7)
plt.title('Correlation heatmap of top 30 features')
plt.xticks(range(len(corr_top.columns)), corr_top.columns, rotation=90, fontsize=8)
plt.yticks(range(len(corr_top.index)), corr_top.index, fontsize=8)
plt.show()

**7.** Looking at the heat map above (last column) we can see which features correlate most (most red) with sale price.  
* OverallQual
* GrLivArea
* GarageCars
* YearBuilt
* FullBath
* TotalBsmtSF
* GarageYrBlt
* 1stFlrSF
* YearRemoddAdd
* TotRmsAbvGrd
* Fireplaces
* Foundation
* OpenPorchSF
* LotArea
* MasVnrArea
* LotFrontage
* WoodDeckSF
* HalfBath
* SaleCondition

# 4. Prepare the Data for ML (15 pts)

Use the Scikit Learn data pipeline and preprocessing API. Write functions for any custom data transformations used that are not available in the Scikit Learn API.

Minimum expectations. Some of these may or may not be necessary for this task. If a step is not necessary, discuss why.

1. Clean the data.
2. Perform feature selection.
3. Perform feature engineering.
5. Standardize or normalize features.

Your data preparation pipeline does not need to use all of the features, but you must use at least one categorical feature, one ordinal feature, and one numeric feature. These can be raw features, or features that you create yourself.

**1.** Cleaning Data
* Load the dataframe again

In [ ]:
df_train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
df_test  = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

X_train_raw = df_train.drop(columns=['SalePrice']).copy()
y_train = df_train['SalePrice'].copy()
X_test_raw  = df_test.copy()

* Remove features with high count of missing values

In [ ]:
#This will be used later on
drop_ls = [
    'Id', 'Street', 'Alley', 'Condition2', 'RoofMatl', 'BsmtFinType2', 'Heating',
    'GarageQual', 'GarageCond', 'PoolQC', 'MiscFeature', 'Condition1', 'Utilities',
    'Exterior2nd','ExterCond', 'FireplaceQu', 'PoolArea', 'Fence', '3SsnPorch',
    'LowQualFinSF', 'MiscVal', 'MSZoning', 'LotShape', 'LandContour', '1stFlrSF',
    '2ndFlrSF'
]

**2.** Feature selection will be conducted with dropping unneeded features together with a feature engineering function

**3.** Feature engineering and cleaning data:

**Binary Indicators**
* We will first add binary indicators for nominal features listed here
* HasLowQualSF: 1 if low-quality finished square footage exists.
* HasMiscFeature: 1 if miscellaneous features (e.g., shed) exist.
* Unfinished: 1 if the house style is unfinished (1.5Unf, 2.5Unf).
* Residential: 1 if zoning type is residential (RL, RM, RH).
* RegShape: 1 if lot shape is regular.
* IsLevel: 1 if land contour is level.
* BsmtFinType1_Unf: 1 if basement finish type is “Unfinished.”
* For HasWoodDeck, HasOpenPorch, HasEnclosedPorch, Has3SsnPorch, HasScreenPorch, 1 if each corresponding area equals zero (no deck/porch).

**Feature Engineering**
* YearsSinceRemodel = YrSold − YearRemodAdd (how long since remodel).
* Total_Home_Quality = OverallQual + OverallCond (combined quality score).
* TotalSF = TotalBsmtSF + 1stFlrSF + 2ndFlrSF (total floor area).
* YrBltAndRemod = YearBuilt + YearRemodAdd (combined construction/remodel age).

**Drop high missing value count feature**
* Drop the features in the array above

In [ ]:
def add_features_and_drop(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['HasLowQualSF']       = (df.get('LowQualFinSF', 0).fillna(0).astype(float) > 0).astype(int)
    df['HasMiscFeature']     = (df.get('MiscVal', 0).fillna(0).astype(float) > 0).astype(int)
    df['Unfinished']         = df.get('HouseStyle', '').isin(['1.5Unf', '2.5Unf']).astype(int)
    df['Residential']        = df.get('MSZoning', '').isin(['RL','RM','RH']).astype(int)
    df['RegShape']           = (df.get('LotShape', '') == 'Reg').astype(int)
    df['IsLevel']            = (df.get('LandContour', '') == 'Lvl').astype(int)
    df['BsmtFinType1_Unf']   = (df.get('BsmtFinType1', '') == 'Unf').astype(int)
    df['HasWoodDeck']        = (df.get('WoodDeckSF', 0).fillna(0).astype(float) == 0).astype(int)
    df['HasOpenPorch']       = (df.get('OpenPorchSF', 0).fillna(0).astype(float) == 0).astype(int)
    df['HasEnclosedPorch']   = (df.get('EnclosedPorch', 0).fillna(0).astype(float) == 0).astype(int)
    df['Has3SsnPorch']       = (df.get('3SsnPorch', 0).fillna(0).astype(float) == 0).astype(int)
    df['HasScreenPorch']     = (df.get('ScreenPorch', 0).fillna(0).astype(float) == 0).astype(int)

    df['YearsSinceRemodel']  = (
        df.get('YrSold', df.get('YearRemodAdd', 0)).fillna(0).astype(float)
        - df.get('YearRemodAdd', 0).fillna(0).astype(float)
    )
    df['Total_Home_Quality'] = df.get('OverallQual', 0).fillna(0).astype(float) \
                             + df.get('OverallCond', 0).fillna(0).astype(float)
    df['TotalSF']            = df.get('TotalBsmtSF', 0).fillna(0).astype(float) \
                             + df.get('1stFlrSF',    0).fillna(0).astype(float) \
                             + df.get('2ndFlrSF',    0).fillna(0).astype(float)
    df['YrBltAndRemod']      = df.get('YearBuilt', 0).fillna(0).astype(float) \
                             + df.get('YearRemodAdd', 0).fillna(0).astype(float)

    return df.drop(columns=[c for c in drop_ls if c in df.columns], errors='ignore')

**Function Transformer**
* Use a function transformer to plug the "add_features_and_drop" function into sklearn and apply it before any preprocessing

In [ ]:
from sklearn.preprocessing import FunctionTransformer
add_feats = FunctionTransformer(
    add_features_and_drop,
    feature_names_out=None,
    validate=False
)

**Handling Skew**
* Function to handle skeweness of training data

In [ ]:
def safe_log1p(X):
    X = np.asarray(X, dtype=float)
    return np.log1p(np.clip(X, 0, None))

**Preprocessing Pipelines**
* num_pipeline fills in the missing numerical features with the column mean and standarizes to zero mean
* cat_pipeline fills in categorical features with the mode
* Columnwise applies both pipelines to their respective dtypes (num > numerical, cat > objects)
* Lastly drop everything else not used

In [ ]:
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='mean')),
    ('log', FunctionTransformer(safe_log1p, feature_names_out=None)),
    ('scale', StandardScaler())
])

ohe = OneHotEncoder(handle_unknown='ignore')

cat_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('ohe', ohe)
])

columnwise = ColumnTransformer(
    transformers=[
        ('cat', cat_pipeline, selector(dtype_include='object')),
        ('num', num_pipeline, selector(dtype_include=np.number)),
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

* Apply all pipelines

In [ ]:
full_pipeline = Pipeline(steps=[
    ('add_feats', add_feats),
    ('prep', columnwise)
])

* The following code is to print names of features after running the model

In [ ]:
full_pipeline.fit(X_train_raw)

X_tmp = full_pipeline.named_steps['add_feats'].transform(X_train_raw)

cat_cols = X_tmp.select_dtypes(include='object').columns
num_cols = X_tmp.select_dtypes(exclude='object').columns

prep = full_pipeline.named_steps['prep']
if not hasattr(prep, "transformers_"):
    prep.fit(X_tmp)

cat_feature_names = (
    prep.named_transformers_['cat']
        .named_steps['ohe']
        .get_feature_names_out(cat_cols)
)
num_feature_names = num_cols.to_numpy()

feat_names = np.concatenate([cat_feature_names, num_feature_names])


# 5. Explore Different Models (15 pts)

Minimum expectations:

1. Train an unregularized linear model, a linear model with default parameters, and a decision tree.
2. Measure and compare their performance.
3. Analyze the most significant variables for each algorithm.
4. Analyze the types of errors the models make.

* First, we split the full training set into a training set (80%) and a validation set (20%)

In [ ]:
from sklearn.model_selection import train_test_split
import scipy.sparse as sp

X_full = df_train.drop(columns=['SalePrice'])
y_full = df_train['SalePrice']
X_tr_raw, X_va_raw, y_tr, y_va = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42
)

full_pipeline.fit(X_tr_raw)
X_tr = full_pipeline.transform(X_tr_raw)
X_va = full_pipeline.transform(X_va_raw)

* Define functions to help us measure performance of our models

In [ ]:
from sklearn.metrics import mean_squared_error

def log_rmse(y_true, y_pred, clip_min=1.0):
    y_pred = np.clip(np.asarray(y_pred).ravel(), clip_min, None)
    y_true = np.asarray(y_true).ravel()
    return np.sqrt(mean_squared_error(np.log(y_true), np.log(y_pred)))

def evaluate(y_true, y_pred, name):
    return {
        "Model": name,
        "Log RMSE": log_rmse(y_true, y_pred),
    }

def to_dense(X):
    return X.toarray() if sp.issparse(X) else X

**Train Models**
* Linear Regression
* Ridge
* DescisionTree Regressor

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor

lin = LinearRegression()
lin.fit(X_tr, y_tr)

ridge = Ridge(random_state=42)
ridge.fit(X_tr, y_tr)

tree = DecisionTreeRegressor(random_state=42)
tree.fit(to_dense(X_tr), y_tr)

In [ ]:
lin_pred   = lin.predict(X_va)
ridge_pred = ridge.predict(X_va)
tree_pred  = tree.predict(to_dense(X_va))

results = pd.DataFrame([
    evaluate(y_va, lin_pred,   "LinearRegression (unregularized)"),
    evaluate(y_va, ridge_pred, "Ridge (default)"),
    evaluate(y_va, tree_pred,  "DecisionTree (default)")
])

results

* Function to show top coefficient of LinearRegression and Ridge

In [ ]:
def top_coefficients(model, names, top_k=15):
    coefs = np.asarray(model.coef_).ravel()
    idx = np.argsort(np.abs(coefs))[::-1][:top_k]
    out = pd.DataFrame({
        "feature": names[idx],
        "coefficient": coefs[idx]
    })
    return out
    
print("LinearRegression")
display(top_coefficients(lin, feat_names, top_k=20))
print("Ridge)")
display(top_coefficients(ridge, feat_names, top_k=20))

**Linear Regression (unregularized)**
* Shows very large (unstable) coefficients → clear multicollinearity
* Duplicated features (BsmtFinType1_Unf) indicate overfitting.
* Directionally correct: newer homes, higher quality, and upscale neighborhoods increase price.
* Not reliable for interpretation due to extreme coefficient magnitudes.

**Ridge Regression (regularized)**
* Coefficients are smaller, smoother, and more realistic.
* Ridge handles correlation between features effectively, giving stable, interpretable results.
* Indicates that location, quality, and living area are primary drivers of house prices.

In [ ]:
def error_report(name, y_true, y_pred, top_n=10):
    df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred})
    df["error"] = df["y_pred"] - df["y_true"]

    print(f"\n Error Analysis of {name} Model")
    print("Top 10 over-predictions")
    display(df.sort_values("error", ascending=False).head(top_n))
    print("Top 10 under-predictions")
    display(df.sort_values("error", ascending=True).head(top_n))
    
error_report("LinearRegression", y_va, lin_pred)
error_report("Ridge", y_va, ridge_pred)
error_report("DecisionTree", y_va, tree_pred)

**Linear Regression**
* Overestimates prices for cheap homes
* Heavyly underestimates expensive homes by up to $270k (Id = 691)
* High magnitude of both over- and under-predictions
* Meaning that linear relationships alone can’t capture nonlinear effects of features

**Ridge Regression**
* Overall behavior similar to Linear Regression but slightly reduced extremes.
* Regularization shrinks coefficients → less overfitting to noise.
* More consistent, but limited improvement for outlier houses.

**Decision Tree**
* Captures nonlinear relationships better, but produces larger local errors.
* Example: predicted $745k for a 403k house
* Handles mid-range homes better but fails on price extremes due to lack of smoothing.

# 6. Fine-tune the Models (15 pts)

1. Fine-tune the hyperparameters for a linear model using RandomizedSearchCV.
2. Fine-tune the hyperparameters for a decision tree using GridSearchCV.

**1.** RandomizedSearchCV for Ridge

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer
from scipy.stats import loguniform

log_rmse_scorer = make_scorer(log_rmse, greater_is_better=False)
ridge_param_dist = {
    "alpha": loguniform(1e-3, 1e3),
    "fit_intercept": [True, False],
    "solver": ["auto", "lbfgs", "sag", "saga", "lsqr"],
    "max_iter": [None, 1000, 5000, 10000],
    "tol": [1e-4, 1e-3, 1e-2],
}

ridge_rs = RandomizedSearchCV(
    estimator=Ridge(random_state=42),
    param_distributions=ridge_param_dist,
    n_iter=40, #adjust for accuracy <> speed
    scoring=log_rmse_scorer,
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

ridge_rs.fit(X_tr, y_tr)
ridge_best = ridge_rs.best_estimator_
ridge_va_pred = ridge_best.predict(X_va)

print("=== Ridge (RandomizedSearchCV) ===")
print("Best params:", ridge_rs.best_params_)
print("Best CV score (neg log-RMSE):", ridge_rs.best_score_)
print("Validation log-RMSE:", log_rmse(y_va, ridge_va_pred))

**2.** GridSearchCV for DescisionTree

In [ ]:
from sklearn.model_selection import GridSearchCV

X_tr_dense = to_dense(X_tr)
X_va_dense = to_dense(X_va)

tree_param_grid = {
    "max_depth": [None, 3, 5, 7, 10, 15, 20],
    "min_samples_split": [2, 5, 10, 20, 50],
    "min_samples_leaf": [1, 2, 5, 10, 20],
    "max_features": [None, "sqrt", "log2", 0.3, 0.5, 0.8],
}

tree_gs = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=tree_param_grid,
    scoring=log_rmse_scorer,
    cv=5,
    n_jobs=-1,
    verbose=1
)

tree_gs.fit(X_tr_dense, y_tr)
tree_best = tree_gs.best_estimator_
tree_va_pred = tree_best.predict(X_va_dense)

print("\n=== DecisionTree (GridSearchCV) ===")
print("Best params:", tree_gs.best_params_)
print("Best CV score (neg log-RMSE):", tree_gs.best_score_)
print("Validation log-RMSE:", log_rmse(y_va, tree_va_pred))

# 7. Summarize your results (15 pts)

1. Provide a 2-3 sentence summary of your solution for a non-technical audience of real estate executives.

2. Provide a technical summary of your solution for the ML team lead at a real estate investment company.

**1. Non-Technical Audience Summary**
* We built a model that estimates home prices in Ames using property characteristics (size, quality, age, neighborhood, etc.).
* After testing several approaches, the tuned Decision Tree performed best on our training data, so we used it to generate the final predictions.
* The model’s pricing is most strongly influenced by location (neighborhood), overall home quality, and living area, and it is generally accurate but can be less precise on very inexpensive or very high-end homes (extremes).

**2. Technical Audience Summary**
* Pipeline: Custom feature engineering (binary flags; TotalSF, Total_Home_Quality, YearsSinceRemodel, YrBltAndRemod) → ColumnTransformer with mean imputation + log1p + scaling for numerics, and most-frequent imputation + OHE for categoricals.
* Models compared: LinearRegression (unregularized), Ridge, and DecisionTreeRegressor on an 80/20 train/validation split; metric: log-RMSE on SalePrice.
* Findings: Ridge reduced coefficient instability vs. OLS but still underpredicted high-price homes; Decision Tree captured nonlinearities best and achieved the lowest validation log-RMSE.
* Tuning: Ridge tuned via RandomizedSearchCV; Decision Tree via GridSearchCV. We refit the tuned Decision Tree on all training data and used it for test predictions/submission.
* Error patterns: Overprediction on some low-price homes and underprediction on some luxury homes; suggests remaining heteroscedasticity/nonlinear effects.
* Next steps: Consider log-target training end-to-end, calibrated tree ensembles (GBM/XGBoost/LightGBM), and interaction features (e.g., Neighborhood × Quality).

# 9. Submit your predictions to the competition

In [ ]:
full_pipeline.fit(X_train_raw)
X_train_all = full_pipeline.transform(X_train_raw)
X_test_all  = full_pipeline.transform(X_test_raw)

X_train_all = to_dense(X_train_all)
X_test_all  = to_dense(X_test_all)

tree_best.fit(X_train_all, y_train)

test_predictions = np.clip(tree_best.predict(X_test_all), 1.0, None)

In [ ]:
final_df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv")
submission = pd.DataFrame({
    'Id': final_df["Id"],
    'SalePrice': test_predictions
})

submission.to_csv('submission.csv', index=False)

In [ ]:
%%bash

cat submission.csv | head -n 5